 # Notebook 3: CNN Model for Human Activity Recognition

In [17]:
eval_loader = DataLoader(train_ds,batch_size=64, shuffle=False)
tr_preds = []
with torch.no_grad():
    for X_batch, y_batch in eval_loader:
        out = model(X_batch)
        p = torch.argmax(out, dim=1)
        tr_preds.append(p)
tr_preds = torch.cat(tr_preds)
correct = (tr_preds == y_train_t).sum().item()
print(correct / len(y_train_t))


0.9975516866158868


In [16]:
model.eval()
train_preds = []
with torch.no_grad():
    for X_batch, y_batch in train_loader:
        out = model(X_batch)
        p = torch.argmax(out, dim=1)
        train_preds.append(p)
train_preds = torch.cat(train_preds)
correct = (train_preds ==y_train_t).sum().item()
print(correct / len(y_train_t))

0.1692056583242655


In [46]:
model.eval()
all_preds = []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        output = model(X_batch)
        preds = torch.argmax(output, dim=1)
        all_preds.append(preds)
all_preds = torch.cat(all_preds)
print(classification_report(y_test_t,all_preds))    

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       496
           1       0.00      0.00      0.00       471
           2       1.00      0.00      0.00       420
           3       0.30      0.63      0.41       491
           4       0.18      0.67      0.29       532
           5       1.00      0.00      0.00       537

    accuracy                           0.23      2947
   macro avg       0.41      0.22      0.12      2947
weighted avg       0.41      0.23      0.12      2947



c:\Users\M Mayavan\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\M Mayavan\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\M Mayavan\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _

In [45]:
for epoch in range(50):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(epoch+1, total_loss)    

1 53.06909231841564
2 16.397325852885842
3 13.132272837683558
4 11.971202129498124
5 11.375040544196963
6 10.515607049688697
7 11.209646757692099
8 10.048042800277472
9 9.108394799754024
10 8.694969495758414
11 8.467590327374637
12 8.966624663677067
13 9.771927959751338
14 7.967945935670286
15 7.372608598321676
16 7.290981293655932
17 7.089805103838444
18 6.521783652715385
19 6.373845126014203
20 6.511388193350285
21 6.947045874549076
22 6.875024997163564
23 6.060391646344215
24 6.149388941936195
25 6.721795893739909
26 6.285850887652487
27 5.478606343735009
28 5.303806237876415
29 5.2241722284816206
30 4.961282148491591
31 5.50153725547716
32 5.325505076441914
33 6.048440455459058
34 4.755541916470975
35 4.437807457288727
36 6.276727143675089
37 5.017494216328487
38 4.6774309440515935
39 4.111756612779573
40 4.140819528256543
41 4.393000161275268
42 4.652435142779723
43 4.290765368845314
44 3.773344109999016
45 4.8322589616291225
46 4.108467213110998
47 5.113739107735455
48 4.76312417

In [44]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001, weight_decay=1e-3)

In [43]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv1d(9, 32,kernel_size=9, padding=4)
        self.conv2 = nn.Conv1d(32, 64,kernel_size=5, padding=2)
        self.pool = nn.MaxPool1d(2)
        self.relu = nn.ReLU()
        self.drop = nn.Dropout(0.7)
        self.fc1 = nn.Linear(2048, 64)
        self.fc2 = nn.Linear(64, 6)
    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.pool(self.relu(self.conv1(x))) 
        x = self.pool(self.relu(self.conv2(x)))                     
        x = x.view(x.size(0), -1)
        x = self.drop(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [36]:
train_ds = TensorDataset(X_train_t, y_train_t)
test_ds = TensorDataset(X_test_t, y_test_t)
train_loader = DataLoader(train_ds, batch_size=64,shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

In [35]:
mean = X_train_t.mean(dim=(1,2), keepdim=True)
std = X_train_t.std(dim=(1,2), keepdim=True) + 1e-8
X_train_t = (X_train_t - mean) / std
mean = X_test_t.mean(dim=(1,2), keepdim=True)
std = X_test_t.std(dim=(1,2), keepdim=True) + 1e-8
X_test_t = (X_test_t - mean) / std

In [3]:
X_train_t = torch.tensor(X_train,dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

In [2]:
base = r"C:\Users\M Mayavan\OneDrive"
base = base + r"\Desktop\HAR_Project\UCI HAR Dataset"
X_train = np.load(base + r"\X_train.npy")
X_test = np.load(base + r"\X_test.npy")
y_train = np.load(base + r"\y_train.npy")
y_test = np.load(base + r"\y_test.npy")

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,TensorDataset
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report